<a href="https://colab.research.google.com/github/nisaerdemli/imageprocessing/blob/main/hafta3g%C3%B6rev3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# -------------------------------------------------------------
# 1. CLAHE'nin Sıfırdan (From Scratch) Gerçeklenmesi
# -------------------------------------------------------------

def clahe_scratch(img, clip_limit=2.0, grid_size=(8, 8)):
    """
    CLAHE algoritmasının sıfırdan NumPy ile gerçeklenmesi.
    img: Tek kanallı (grayscale) uint8 formatında görüntü.
    clip_limit: Kontrast sınırlama eşiği.
    grid_size: (grid_y, grid_x) blok sayısı.
    """
    h, w = img.shape
    grid_y, grid_x = grid_size

    # Blok boyutlarını hesaplama
    tile_h = h / grid_y
    tile_w = w / grid_x

    # Her blok için dönüşüm fonksiyonlarını (CDF tabloları) tutacak dizi
    # Boyut: [grid_y, grid_x, 256]
    mappings = np.zeros((grid_y, grid_x, 256), dtype=np.float32)

    # Adım 1 & 2 & 3: Blok bazlı Histogram, Kırpma (Clipping) ve Dağıtım
    for r in range(grid_y):
        for c in range(grid_x):
            r_start = int(round(r * tile_h))
            r_end = int(round((r + 1) * tile_h))
            c_start = int(round(c * tile_w))
            c_end = int(round((c + 1) * tile_w))

            tile = img[r_start:r_end, c_start:c_end]

            # Histogram çıkarma
            hist, _ = np.histogram(tile.flatten(), bins=256, range=[0, 256])

            # Clip Limit Uygulama
            num_pixels = tile.size
            # Normalize edilmiş limitin piksel karşılığı
            actual_clip = max(1, int(clip_limit * (num_pixels / 256.0)))

            excess = np.sum(np.maximum(hist - actual_clip, 0))
            hist = np.minimum(hist, actual_clip)

            # Kırpılan fazlalığı tüm bin'lere eşit dağıt
            hist += excess // 256
            remainder = excess % 256
            if remainder > 0:
                hist[:remainder] += 1

            # CDF (Kümülatif Dağılım) hesaplama ve normalize etme [0, 255]
            cdf = hist.cumsum()
            cdf_normalized = (cdf / cdf[-1]) * 255.0
            mappings[r, c] = cdf_normalized

    # Adım 4: Bilineer İnterpolasyon ile Piksellerin Hesaplanması
    out = np.zeros_like(img, dtype=np.uint8)

    # Blok merkez koordinatları
    # Piksellerin hangi 4 blok merkezinin arasında kaldığını bulmak için
    for y in range(h):
        for x in range(w):
            # Koordinatın blok indislerindeki konumu (merkez kaydırmalı)
            # Blok merkezi: tile_w * (c + 0.5)
            c = (x - tile_w / 2.0) / tile_w
            r = (y - tile_h / 2.0) / tile_h

            # En yakın 4 grid noktası
            c1 = int(np.floor(c))
            r1 = int(np.floor(r))
            c2 = c1 + 1
            r2 = r1 + 1

            # Ağırlıklar (interpolar kesirler)
            ax = c - c1
            ay = r - r1

            # Köşe/Kenar durumlarında sınır kontrolü (Clamping)
            c1 = np.clip(c1, 0, grid_x - 1)
            c2 = np.clip(c2, 0, grid_x - 1)
            r1 = np.clip(r1, 0, grid_y - 1)
            r2 = np.clip(r2, 0, grid_y - 1)

            val = img[y, x]

            # 4 merkezden gelen dönüştürülmüş yoğunluklar
            v11 = mappings[r1, c1, val]
            v12 = mappings[r1, c2, val]
            v21 = mappings[r2, c1, val]
            v22 = mappings[r2, c2, val]

            # 2 Boyutlu (Bilinear) enterpolasyon formülü
            interpolated = (1 - ay) * ((1 - ax) * v11 + ax * v12) + \
                           ay * ((1 - ax) * v21 + ax * v22)

            out[y, x] = np.clip(np.round(interpolated), 0, 255)

    return out


# -------------------------------------------------------------
# 2. Test ve Karşılaştırma
# -------------------------------------------------------------

# Düşük kontrastlı görüntüyü grayscale olarak oku
# 'input.jpg' yerine kendi resim dosyanın yolunu yazabilirsin
img_path = 'input.jpg'
img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

# Eğer elinde düşük kontrastlı resim yoksa test için sentetik oluşturabilirsin:
if img is None:
    # Test amaçlı düşük kontrastlı yapay bir gri resim üretelim
    x = np.linspace(0, 1, 256)
    y = np.linspace(0, 1, 256)
    xx, yy = np.meshgrid(x, y)
    img = (np.sin(xx * 3 * np.pi) * 20 + 120).astype(np.uint8)

# A. OpenCV Hazır Fonksiyonu ile CLAHE
clahe_cv = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
img_clahe_opencv = clahe_cv.apply(img)

# B. Sıfırdan Yazdığımız Fonksiyon ile CLAHE
img_clahe_scratch = clahe_scratch(img, clip_limit=2.0, grid_size=(8, 8))

# -------------------------------------------------------------
# 3. Sonuçların Görselleştirilmesi
# -------------------------------------------------------------
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(img, cmap='gray')
plt.title('Orijinal (Düşük Kontrast)')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(img_clahe_opencv, cmap='gray')
plt.title('OpenCV CLAHE')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(img_clahe_scratch, cmap='gray')
plt.title('Sıfırdan (Custom) CLAHE')
plt.axis('off')

plt.tight_layout()
plt.show()